In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.express as px
import patsy
from scipy import stats
import warnings
warnings.simplefilter("ignore", FutureWarning)
METABOTKIT = r'C:\Users\jaileru\Projects\metabolomics-toolkit'
ROOT = r"\\172.22.48.112\Sumner_HHEAR_Lichtveld\4a. Data_LC-MS_OrbiTrap\5-Statistical Analyses\Julian Analysis\July-2026"
import sys,os
sys.path.append(os.path.abspath(METABOTKIT))
from differential_abundance import DifferentialAbundance
from dimensionality_reduction import pca_plot
PROJECT_ID = 'ST002118'
from itertools import chain
from docx import Document
sys.path.append(os.path.abspath(os.path.join(ROOT, "Analysis")))
from ST002118 import *

In [2]:
untargeted_data = clean_untargeted_reference(rf'{ROOT}/Data/Untargeted/reference_copy_untargeted_data.parquet')
categorical_data = clean_categorical_reference(rf'{ROOT}/Data/Untargeted/reference_copy_categorical_data.parquet',
                                               additional_cols=['v113a','v113b','v113c','v113d','v108a','v107g'])
ethnicity_map = {"v113a":'Ethnicity9Categories',
                 "v113b":"Ethnicity8Categories",
                 'v113c':"Ethnicity7Categories",
                 'v113d':"Ethnicity6Categories",
                 'v108a':"DistrictAtIntake10Categories",
                 'v107g':'MaternalAgeAtIntake3Categories'}
categorical_data = categorical_data.rename(columns=ethnicity_map)

In [56]:
ethnicity_map = {
    "1.0": "Creole",
    "2.0": "Hindustani",
    "3.0": "Indigenous",
    "4.0": "Javanese",
    "5.0": "Maroon",
    "6.0": "Mixed"
}

district_map = {"1.0":"Paramaribo",
                "2.0":"Wanica",
                "3.0":"Commewijne",
                "4.0":"Marowijne",
                "5.0":"Saramacca",
                "6.0":"Coronie",
                "7.0":"Nickerie",
                "8.0":"Para",
                "9.0":"Brokopondo",
                "10.0":"Sipaliwini"}
maternal_age_map = {"1.0":"16-19 years",
                    "2.0": "20-34 years",
                    "3.0": "35+ years"}

ptb_map = {"1.0":"yes",
           "2.0":"no"}
lbw_map = {"1.0":"yes",
           "2.0":"no"}

In [58]:
contingency_table = categorical_data[['Ethnicity6Categories','DistrictAtIntake10Categories','PretermBirthLessThan37weeks','MaternalAgeAtIntake3Categories','BirthweightLessThan2500g']].copy()
contingency_table['BirthweightLessThan2500g'] = contingency_table['BirthweightLessThan2500g'].dropna().astype(str).map(lbw_map)
contingency_table['PretermBirthLessThan37weeks'] = contingency_table['PretermBirthLessThan37weeks'].dropna().astype(str).map(ptb_map)
contingency_table['MaternalAgeAtIntake3Categories'] = contingency_table['MaternalAgeAtIntake3Categories'].dropna().astype(str).map(maternal_age_map)
contingency_table['Ethnicity6Categories'] = contingency_table['Ethnicity6Categories'].dropna().astype(str).map(ethnicity_map)
contingency_table['DistrictAtIntake10Categories'] = contingency_table['DistrictAtIntake10Categories'].dropna().astype(str).map(district_map)
contingency_table = contingency_table.dropna()

In [65]:
contingency_table_all_districts = pd.crosstab(contingency_table['Ethnicity6Categories'],contingency_table['DistrictAtIntake10Categories'])
contingency_table_large_districts = contingency_table_all_districts[['Nickerie','Paramaribo','Wanica']].copy()


contingency_table_maternal_age_ptb = pd.crosstab(contingency_table['PretermBirthLessThan37weeks'],contingency_table['MaternalAgeAtIntake3Categories'])
contingency_table_maternal_age_lbw = pd.crosstab(contingency_table['BirthweightLessThan2500g'],contingency_table['MaternalAgeAtIntake3Categories'])
contingency_table_ptb_ethnicity = pd.crosstab(contingency_table['PretermBirthLessThan37weeks'],contingency_table['Ethnicity6Categories'])
contingency_table_lbw_ethnicity = pd.crosstab(contingency_table['BirthweightLessThan2500g'],contingency_table['Ethnicity6Categories'])

In [84]:
from scipy.stats import chi2_contingency,fisher_exact
#print(contingency_table_all_districts)
print("-"*10)
odds,p = fisher_exact(contingency_table_all_districts)
print(odds,p)
print("-"*10)

#print(contingency_table_large_districts)
print("-"*10)
odds,p = fisher_exact(contingency_table_large_districts)
print(odds,p)
print("-"*10)

----------
1.7980684596596313e-125 0.0001
----------
----------
9.142191371115277e-53 0.0001
----------


In [75]:
#print(contingency_table_ptb_ethnicity)
print("-"*10)
chi2,p,dof,expected = chi2_contingency(contingency_table_ptb_ethnicity)
print(chi2,p)
print("-"*10)

#print(contingency_table_lbw_ethnicity)
print("-"*10)
chi2,p,dof,expected = chi2_contingency(contingency_table_lbw_ethnicity)
print(chi2,p)
print("-"*10)


----------
7.5706722936196265 0.1815380348842837
----------
----------
9.748423460225705 0.08268558287829235
----------


In [78]:
contingency_table_maternal_age_ptb

MaternalAgeAtIntake3Categories,16-19 years,20-34 years,35+ years
PretermBirthLessThan37weeks,,,
no,62,433,89
yes,8,48,2


In [79]:

#print(contingency_table_maternal_age_lbw)
print("-"*10)
chi2,p,dof,expected = chi2_contingency(contingency_table_maternal_age_lbw)
print(chi2,p)
print("-"*10)

#print(contingency_table_maternal_age_ptb)
print("-"*10)
chi2,p,dof,expected = chi2_contingency(contingency_table_maternal_age_ptb)
print(chi2,p)
print('-'*10)

----------
4.120813926313657 0.12740211137785062
----------
----------
6.186197595125323 0.04536117130104164
----------
